In [1]:
# Common imports for analysis and modeling
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [ ]:
dataset = pd.read_csv('drive/MyDrive/Labs/CSE422/Dataset/8.csv')
#csv read directly from github loaded onto volunteer dataframe. It can be read from google drive or locally as well

display(dataset)

In [ ]:
dataset.isnull().sum() #number of null values in the dataframe

# Introduction

The CSV ("8.csv") contains the UCI Heart Disease dataset, combining patient records from Cleveland, Hungary, Switzerland, and VA Long Beach sources. It focuses on predicting heart disease presence (target "num": 0=no disease, 1-4=severity levels) using clinical measurements.

# Dataset Description

There are 16 features in the given dataset, and 920 data points. Features are of the following types:

| Type               | Examples                                                                                                                                                             |
| ------------------ | -------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Numeric            | id, age, trestbps (resting BP), chol (cholesterol), thalch (max heart rate), oldpeak (ST depression), ca (vessels), num (target)​                                  |
| Categorical (text) | sex (Male/Female), dataset (Cleveland/etc.), cp (chest pain type), restecg (ECG results), exang (exercise angina), slope (ST segment), thal (thallium stress) |
| Boolean            | fbs (>120mg/dl blood sugar)​                                                                                                                                   |

In [ ]:
dataset.info()

# Data Analysis

### Feature Type

In [ ]:
#Selecting Numerical Features
numerical_data = dataset.select_dtypes(include='number')
numerical_features=numerical_data.columns.tolist()

print(f'There are {len(numerical_features)} numerical features:')
print(numerical_features, "\n")


# Selecting Categorical Features
categorical_data=dataset.select_dtypes(include='object')
categorical_features=categorical_data.columns.tolist()

print(f'There are {len(categorical_features)} categorical features:')
print(categorical_features)

# fbs is defined as 'object' in the given dataset, and not detected as 'bool'

### Variance

In [ ]:
display(numerical_data.var())
display(categorical_data.describe().T)

### Skewness

In [ ]:
numerical_data.skew()

### Outlier Detection

In [ ]:
numerical_data.hist(figsize=(12,12),bins=20)
plt.show()

In [ ]:
# Select only numerical columns for boxplot analysis
numeric_cols = dataset.select_dtypes(include=['int64', 'float64']).columns

# Set up the figure
plt.figure(figsize=(20, 30))

# Plot boxplots for each numerical feature including the target variable 'OUTCOME'
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(len(numeric_cols), 1, i)
    sns.boxplot(x=dataset[col], color='skyblue')
    plt.title(f'Boxplot of {col}', fontsize=12)
    plt.tight_layout()

plt.show()

### Correlation Matrix

In [ ]:
# Calculate the correlation matrix
correlation_matrix = numerical_data.corr()
display(correlation_matrix)

# Plotting the heatmap for correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.3f', linewidths=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(3,1, figsize=(10, 10))
## Correlation coefficient using different methods
corr1 = numerical_data.corr('pearson')[['num']].sort_values(by='num', ascending=False)
corr2 = numerical_data.corr('spearman')[['num']].sort_values(by='num', ascending=False)
corr3 = numerical_data.corr('kendall')[['num']].sort_values(by='num', ascending=False)

#setting titles for each plot
ax[0].set_title('Pearson Method')
ax[1].set_title('Spearman Method')
ax[2].set_title('Kendall Method')

## Generating heatmaps of each methods
sns.heatmap(corr1, ax=ax[0], annot=True)
sns.heatmap(corr2, ax=ax[1], annot=True)
sns.heatmap(corr3, ax=ax[2], annot=True)

plt.show()

In [ ]:
X = df.drop(columns=['num', 'id'])
y = (df['num'] > 0).astype(int)  # Binary classification: 0=no disease, 1=disease

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Target distribution:\n{y.value_counts()}")

## Best Performing Model

Based on the evaluation metrics above:

**Best Model**: {best_model}

**Performance Metrics**:
- Accuracy: {best_accuracy}
- Precision: {best_precision}
- Recall: {best_recall}
- F1-Score: {best_f1}
- ROC-AUC: {best_roc}

## Analysis and Probable Reasons

1. **Feature Importance**: The correlation analysis revealed that features like thalch (max heart rate), oldpeak (ST depression), and ca (number of major vessels) have strong correlation with the target variable, contributing to model performance.

2. **Data Quality**: The KNN imputation successfully handled missing values in ca and thal columns while preserving feature relationships, improving model training efficiency.

3. **Model Characteristics**:
   - **Scikit-learn models** benefit from well-preprocessed, scaled data and explicit feature engineering via ColumnTransformer.
   - **Neural Network** provides non-linear decision boundaries but requires careful tuning of architecture and learning rate.
   - **K-Means Clustering** demonstrates the inherent separability of disease vs. non-disease cases in the feature space.

4. **Class Balance**: The dataset is imbalanced with more disease-positive cases. The use of `class_weight='balanced'` in sklearn models helps mitigate this bias.

5. **Evaluation Metrics**: F1-Score is preferred over accuracy for imbalanced datasets, providing a harmonic mean of precision and recall. ROC-AUC measures the probability of correct disease classification across all thresholds.

### Recommendations for Further Improvement:
- Perform hyperparameter tuning using GridSearchCV or RandomizedSearchCV.
- Investigate feature selection techniques (e.g., recursive feature elimination).
- Apply ensemble methods or stacking for potentially better generalization.
- Collect more balanced training data if possible.


# Conclusion

In [ ]:
# ROC-AUC comparison (excluding NaN values)
roc_scores = results_df['ROC-AUC'].dropna().sort_values(ascending=False)

if len(roc_scores) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    roc_scores.plot(kind='bar', ax=ax, color='mediumseagreen', edgecolor='black')
    ax.set_title('Model ROC-AUC Score Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel('ROC-AUC', fontsize=12)
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(model_results).T
results_df = results_df.sort_values(by='F1-Score', ascending=False)

print("="*70)
print("COMPREHENSIVE MODEL PERFORMANCE COMPARISON")
print("="*70)
print(results_df.round(4))

# Bar chart for Accuracy and F1-Score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
results_df['Accuracy'].sort_values(ascending=False).plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# F1-Score comparison
results_df['F1-Score'].sort_values(ascending=False).plot(kind='bar', ax=axes[1], color='lightcoral', edgecolor='black')
axes[1].set_title('Model F1-Score Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('F1-Score', fontsize=12)
axes[1].set_xlabel('Model', fontsize=12)
axes[1].set_ylim([0, 1])
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Model Evaluation

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# K-Means clustering on preprocessed data
print("Applying K-Means Clustering...")
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_preprocessed)

# Evaluate clustering
silhouette = silhouette_score(X_preprocessed, cluster_labels)
davies_bouldin = davies_bouldin_score(X_preprocessed, cluster_labels)

print(f"\nK-Means Clustering Results:")
print(f"Silhouette Score: {silhouette:.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin:.4f}")
print(f"Cluster distribution:\n{pd.Series(cluster_labels).value_counts()}")

# Compare clusters with actual disease labels
comparison = pd.DataFrame({'Actual': y, 'Predicted_Cluster': cluster_labels})
print(f"\nCluster vs Actual Disease Label Cross-tabulation:")
print(pd.crosstab(comparison['Predicted_Cluster'], comparison['Actual'], margins=True))

## Unsupervised Learning: K-Means Clustering

In [ ]:
# Reshape data for neural network (from 2D to required format)
X_train_nn = X_train.toarray() if hasattr(X_train, 'toarray') else X_train
X_test_nn = X_test.toarray() if hasattr(X_test, 'toarray') else X_test
y_train_nn = y_train.reshape(-1, 1).astype(np.float32)
y_test_nn = y_test.reshape(-1, 1).astype(np.float32)

print(f"Neural Network training data shape: {X_train_nn.shape}")
print(f"Neural Network test data shape: {X_test_nn.shape}")

# Create and train Neural Network
print("\nTraining Neural Network...")
nn_model = NeuralNetworkModel()
nn_model.set_layers([
    NeuronLayer(X_train_nn.shape[1], 64, "RELU"),
    NeuronLayer(64, 32, "RELU"),
    NeuronLayer(32, 1, "RELU")
])
nn_model.compile(loss="MSE", optimizer="GD", learningRate=0.01)

training_error = nn_model.fit(X_train_nn, y_train_nn, epochs=50, batchSize=32)

# Plot loss curve
plt.figure(figsize=(10, 6))
plt.plot(training_error, marker='o')
plt.title("Neural Network Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.grid(True)
plt.tight_layout()
plt.show()

# Evaluate Neural Network
test_error = nn_model.evaluate(X_test_nn, y_test_nn)
print(f"\nNeural Network - Test MSE: {test_error:.4f}")

## Neural Network Training

In [ ]:
# Initialize models
sklearn_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1),
}

# Train and evaluate models
model_results = {}
for name, model in sklearn_models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    print(f"\n{name} - Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No Disease', 'Disease']).plot(cmap='Blues')
    plt.title(f'{name} - Confusion Matrix')
    plt.tight_layout()
    plt.show()
    
    # ROC Curve (if probability available)
    if y_proba is not None:
        roc_auc = roc_auc_score(y_test, y_proba)
        RocCurveDisplay.from_predictions(y_test, y_proba, name=name)
        plt.title(f'{name} - ROC Curve (AUC={roc_auc:.3f})')
        plt.tight_layout()
        plt.show()
    else:
        roc_auc = None
    
    # Store results
    report = classification_report(y_test, y_pred, output_dict=True)
    model_results[name] = {
        'Accuracy': report['accuracy'],
        'Precision': report['weighted avg']['precision'],
        'Recall': report['weighted avg']['recall'],
        'F1-Score': report['weighted avg']['f1-score'],
        'ROC-AUC': roc_auc if roc_auc is not None else np.nan,
    }

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Prepare data for scikit-learn models
numeric_tf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_tf = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocess_pipe = ColumnTransformer([
    ('num', numeric_tf, numeric_features),
    ('cat', categorical_tf, categorical_features),
])

# Fit preprocessing on data
X_preprocessed = preprocess_pipe.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.2, stratify=y, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Target distribution in train set:\n{pd.Series(y_train).value_counts()}")

## Scikit-learn Models (Logistic Regression, Decision Tree, KNN)

In [ ]:
# Neural Network Model
class NeuralNetworkModel:
    def __createBatch(self, X: np.array, Y: np.array, batchSize: int) -> tuple:
        miniX, miniY = np.array([X[:batchSize]]), np.array([Y[:batchSize]])
        for idx in range(1, X.shape[0] // batchSize):
            miniX = np.append(miniX, np.array([X[idx * batchSize : (idx + 1) * batchSize]]), axis=0)
            miniY = np.append(miniY, np.array([Y[idx * batchSize : (idx + 1) * batchSize]]), axis=0)
        return miniX, miniY

    def __forwardPropagation(self, X: np.array) -> np.array:
        output = X
        for layer in self.layers:
            output = layer.forwardPropagation(output)
        return output

    def __backPropagation(self, Y: np.array) -> None:
        gradient = Y
        for layer in self.layers[::-1]:
            gradient = layer.backPropagation(gradient)

    def set_layers(self, layers: list) -> None:
        self.layers = layers

    def compile(self, loss, optimizer, learningRate) -> None:
        self.loss = getLossFunction(loss)()
        for layer in self.layers[::-1]:
            layer.build(optimizer, learningRate)

    def predict(self, X: np.array) -> np.array:
        return self.__forwardPropagation(X)

    def evaluate(self, X: np.array, Y: np.array) -> float:
        output = self.predict(X)
        return self.loss.loss(output, Y)

    def fit(self, X: np.array, Y: np.array, epochs: int, batchSize=None) -> np.array:
        batchSize = (batchSize if batchSize else X.shape[0])
        self.X, self.Y = self.__createBatch(X, Y, batchSize)

        self.error = np.array([])
        for epoch in range(epochs):
            epochError = np.array([])
            for idx in range(self.X.shape[0]):
                epochError = np.append(epochError, self.loss.loss(self.__forwardPropagation(self.X[idx]), self.Y[idx]))
                self.__backPropagation(self.loss.deriv())

            epochError /= epochError.shape[0]
            self.error = np.append(self.error, epochError.sum() / epochError.shape[0])
            if (epoch + 1) % 10 == 0:
                print("Epoch:", epoch + 1, "Error:", round(self.error[-1], 4))

        return self.error

In [ ]:
# Single Layer of Neurons
class NeuronLayer:
    def __init__(self, inputNeurons, outputNeurons, activation, biasFlag=True, randomState=42) -> None:
        np.random.seed(randomState)
        self.inputNeurons = inputNeurons
        self.outputNeurons = outputNeurons
        self.biasFlag = biasFlag

        self.activation = getActivationFunction(activation)()

        self.weights = self.__initParameters((self.inputNeurons, self.outputNeurons))
        self.bias = self.__initParameters((1, self.outputNeurons))

    def __initParameters(self, dimension: tuple) -> np.array:
        return np.random.uniform(-np.sqrt(2/(dimension[0] + dimension[1])), np.sqrt(2/(dimension[0] + dimension[1])), size=(dimension[0], dimension[1]))

    def build(self, optimizer, learningRate) -> None:
        self.learningRate = learningRate
        self.optimizer = getOptimizer(optimizer)(learningRate)

    def forwardPropagation(self, X: np.array) -> np.array:
        self.X = X
        self.output = np.dot(self.X, self.weights) + (self.biasFlag * self.bias)
        self.output = self.activation.forwardPropagation(self.output)
        return self.output

    def backPropagation(self, upstreamGradient: np.array) -> np.array:
        delta = self.activation.backPropagation(upstreamGradient)
        weightGrad = np.dot(self.X.T, delta) / self.X.shape[0]
        biasGrad = np.dot(np.ones((1, self.X.shape[0])), delta) / self.X.shape[0]

        self.weights -= self.optimizer.gradients(weightGrad)
        self.bias -= self.optimizer.gradients(biasGrad)

        downstreamGradient = np.dot(delta, self.weights.T) / self.X.shape[0]
        return downstreamGradient

In [ ]:
# Activation Function (ReLU)
class ReLU:
    def forwardPropagation(self, inp: np.array) -> np.array:
        self.inputAct = np.maximum(0, inp)
        return self.inputAct

    def backPropagation(self, delta: np.array) -> np.array:
        return delta * np.where(self.inputAct > 0, 1, 0)


# Loss Function (Mean Square Error)
class MeanSquareError:
    def loss(self, Y: np.array, y: np.array) -> np.array:
        self.Y = Y
        self.y = y
        return np.absolute((Y - y) ** 2).sum() / y.shape[0]

    def deriv(self) -> np.array:
        return (self.Y - self.y) / self.y.shape[0]


# Optimizer (Gradient Descent)
class GradientDescent:
    def __init__(self, learningRate) -> None:
        self.learningRate = learningRate

    def gradients(self, gradients) -> np.array:
        return self.learningRate * gradients


# Helper functions
def getActivationFunction(activation: str) -> object:
    if activation == "RELU":
        return ReLU
    else:
        raise Exception("Cannot find the activation function.")


def getLossFunction(loss: str) -> object:
    if loss == "MSE":
        return MeanSquareError
    else:
        raise Exception("Cannot find the loss function.")


def getOptimizer(optimizer: str) -> object:
    if optimizer == "GD":
        return GradientDescent
    else:
        raise Exception("Cannot find the optimizer.")

# Model Training

## Neural Network Implementation (Custom)

## Problem 2: Feature Scaling and Encoding for Model Training
**Problem**: Models require numeric inputs; categorical variables must be one-hot encoded and numeric features should be scaled for fair comparison.

**Solution**: Use ColumnTransformer with separate pipelines for numeric (SimpleImputer + StandardScaler) and categorical (SimpleImputer + OneHotEncoder) features.

In [ ]:
numeric_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'thal']
X = df_encode[numeric_cols].copy()

# Apply KNN Imputer
imputer = KNNImputer(n_neighbors=5, weights='distance')
X_imputed = imputer.fit_transform(X)
df_imputed = pd.DataFrame(X_imputed, columns=numeric_cols)

# Round discrete variables
df_imputed['ca'] = df_imputed['ca'].round().clip(0, 3).astype(int)
df_imputed['thal'] = np.round(df_imputed['thal']).clip(3, 7).astype(int)
df_imputed['thal'] = df_imputed['thal'].replace({3:3, 6:6, 7:7})

# Quality Verification histograms
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(X['ca'].dropna(), bins=10, color='blue', edgecolor='black')
axes[0, 0].set_title('ca: Original (n={})'.format(X['ca'].notna().sum()))
axes[0, 0].set_ylabel('Count')

axes[0, 1].hist(df_imputed['ca'], bins=10, color='orange', edgecolor='black')
axes[0, 1].set_title('ca: After Imputation (n={})'.format(len(df_imputed)))

axes[1, 0].hist(X['thal'].dropna(), bins=[3, 6, 7, 8], color='blue', edgecolor='black')
axes[1, 0].set_title('thal: Original (n={})'.format(X['thal'].notna().sum()))
axes[1, 0].set_ylabel('Count')

axes[1, 1].hist(df_imputed['thal'], bins=[3, 6, 7, 8], color='orange', edgecolor='black')
axes[1, 1].set_title('thal: After Imputation (n={})'.format(len(df_imputed)))

plt.tight_layout()
plt.show()

# Update dataframe with imputed values
df[numeric_cols] = df_imputed
print("\n>>> Dataframe updated with imputed values!")
print(f"Final shape: {df.shape}")
print(f"Missing values after imputation:\n{df.isnull().sum()}")

In [ ]:
df = pd.read_csv('drive/MyDrive/Labs/CSE422/Dataset/8.csv')
df_encode = df.copy()

# Encoding thal and ca columns
thal_map = {'normal': 3.0, 'fixed defect': 6.0, 'reversable defect': 7.0}
df_encode['thal'] = df['thal'].map(thal_map)
df_encode['ca'] = pd.to_numeric(df['ca'], errors='coerce')  # Ensure numeric

print(f"Data shape before preprocessing: {df_encode.shape}")
print(f"Missing values after encoding:\n{df_encode.isnull().sum()}")

## Problem 1: Handling Categorical and Missing Values
**Problem**: The dataset contains categorical text columns (thal, ca) and missing values that must be handled before training machine learning models.

**Solution**: Encode categorical columns using mapping; apply KNN imputation (k=5, distance-weighted) to handle numeric missingness while preserving feature relationships.

# Dataset Pre-processing